<a href="https://colab.research.google.com/github/AkashssA/MACHINE_LEARNING_2/blob/main/foil2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Simple FOIL implementation for a small dataset

# ---------------------------------------------------------
# Dataset
# ---------------------------------------------------------

# Facts are represented as:
# predicate(subject, object)

facts = {
    "parent": {
        ("alice", "bob"),
        ("alice", "carol"),
        ("bob", "david"),
        ("carol", "emma")
    },

    "female": {
        ("alice",),
        ("carol",),
        ("emma",)
    },

    "male": {
        ("bob",),
        ("david",)
    }
}

# Target predicate:
# grandparent(X, Y)

# Positive examples
positive_examples = {
    ("alice", "david"),
    ("alice", "emma")
}

# Negative examples
negative_examples = {
    ("alice", "bob"),
    ("bob", "emma"),
    ("carol", "david")
}


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def parent(x, y):
    return (x, y) in facts["parent"]


def female(x):
    return (x,) in facts["female"]


def male(x):
    return (x,) in facts["male"]


# ---------------------------------------------------------
# Candidate rules
# ---------------------------------------------------------

# For this small example, we manually define possible
# literals that FOIL can consider.

candidate_literals = [
    "parent(X, Z)",
    "parent(Z, Y)",
    "female(X)",
    "female(Y)",
    "male(X)",
    "male(Y)"
]


def satisfies_literal(example, literal):
    """
    Check whether a literal is true for an example.

    For the grandparent example:
        X = first element
        Y = second element

    Z is searched for automatically.
    """

    X, Y = example

    if literal == "female(X)":
        return female(X)

    if literal == "female(Y)":
        return female(Y)

    if literal == "male(X)":
        return male(X)

    if literal == "male(Y)":
        return male(Y)

    # parent(X, Z) AND parent(Z, Y)
    if literal == "parent(X, Z)":
        return any(parent(X, z) for z in get_people())

    if literal == "parent(Z, Y)":
        return any(parent(z, Y) for z in get_people())

    return False


def get_people():
    people = set()

    for x, y in facts["parent"]:
        people.add(x)
        people.add(y)

    return people


# ---------------------------------------------------------
# Rule representation
# ---------------------------------------------------------

class Rule:
    def __init__(self):
        self.conditions = []

    def add_condition(self, condition):
        self.conditions.append(condition)

    def covers(self, example):
        """
        Check whether the rule covers an example.
        """

        X, Y = example

        # Special handling for the two conditions needed
        # for grandparent(X,Y).

        has_parent_xz = "parent(X, Z)" in self.conditions
        has_parent_zy = "parent(Z, Y)" in self.conditions

        if has_parent_xz and has_parent_zy:

            # There must exist the SAME Z satisfying both.
            for z in get_people():
                if parent(X, z) and parent(z, Y):
                    return True

            return False

        # If only parent(X,Z) exists
        if has_parent_xz:
            if not any(parent(X, z) for z in get_people()):
                return False

        # If only parent(Z,Y) exists
        if has_parent_zy:
            if not any(parent(z, Y) for z in get_people()):
                return False

        # Other conditions
        if "female(X)" in self.conditions:
            if not female(X):
                return False

        if "female(Y)" in self.conditions:
            if not female(Y):
                return False

        if "male(X)" in self.conditions:
            if not male(X):
                return False

        if "male(Y)" in self.conditions:
            if not male(Y):
                return False

        return True

    def __str__(self):
        if not self.conditions:
            return "grandparent(X, Y) :- TRUE"

        return "grandparent(X, Y) :- " + ", ".join(self.conditions)


# ---------------------------------------------------------
# FOIL Gain
# ---------------------------------------------------------

def foil_gain(rule, literal, pos, neg):
    """
    Simplified FOIL Gain.

    We calculate how many positive and negative examples
    remain after adding a literal.

    gain = positive_remaining / total_remaining
    """

    old_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    old_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Temporarily add literal
    rule.add_condition(literal)

    new_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    new_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Remove temporary literal
    rule.conditions.pop()

    old_total = len(old_covered_pos) + len(old_covered_neg)
    new_total = len(new_covered_pos) + len(new_covered_neg)

    if new_total == 0:
        return 0

    old_probability = (
        len(old_covered_pos) / old_total
        if old_total else 0
    )

    new_probability = (
        len(new_covered_pos) / new_total
        if new_total else 0
    )

    return new_probability - old_probability


# ---------------------------------------------------------
# FOIL algorithm
# ---------------------------------------------------------

def foil(pos, neg):

    learned_rules = []

    pos = set(pos)
    neg = set(neg)

    while pos:

        rule = Rule()
        rule_neg = set(neg)

        print("\nStarting new rule")

        while rule_neg:

            best_literal = None
            best_gain = -1

            for literal in candidate_literals:

                if literal in rule.conditions:
                    continue

                gain = foil_gain(
                    rule,
                    literal,
                    pos,
                    rule_neg
                )

                print(
                    f"  Candidate: {literal:15} "
                    f"Gain = {gain:.3f}"
                )

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            if best_literal is None:
                break

            rule.add_condition(best_literal)

            # Keep only negative examples still covered
            rule_neg = {
                example
                for example in rule_neg
                if rule.covers(example)
            }

            print(
                f"  Added: {best_literal}"
            )

        learned_rules.append(rule)

        # Remove positive examples covered by this rule
        covered_pos = {
            example
            for example in pos
            if rule.covers(example)
        }

        pos -= covered_pos

        print("Learned:", rule)
        print("Covered positive examples:", covered_pos)

    return learned_rules


# ---------------------------------------------------------
# Run FOIL
# ---------------------------------------------------------

rules = foil(
    positive_examples,
    negative_examples
)

print("\n==============================")
print("FINAL LEARNED RULES")
print("==============================")

for rule in rules:
    print(rule)




Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.100
  Candidate: female(Y)       Gain = 0.100
  Candidate: male(X)         Gain = -0.400
  Candidate: male(Y)         Gain = -0.067
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(Y)       Gain = 0.500
  Candidate: male(X)         Gain = 0.000
  Candidate: male(Y)         Gain = -0.167
  Added: female(Y)
Learned: grandparent(X, Y) :- female(X), female(Y)
Covered positive examples: {('alice', 'emma')}

Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.083
  Candidate: female(Y)       Gain = -0.250
  Candidate: male(X)         Gain = -0.250
  Candidate: male(Y)         Gain = 0.083
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Ca

In [2]:
from google.colab import files

uploaded = files.upload()


Saving FOLIO-main.zip to FOLIO-main (1).zip


In [7]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

print("Extracted successfully!")


Extracted successfully!


In [8]:
!find /content/dataset -type d -name "data"


/content/dataset/FOLIO-main/data


In [13]:
DATA_PATH = "/content/dataset/FOLIO-main/data/v0.0"

In [14]:
import os

for root, dirs, files in os.walk(DATA_PATH):
    print("\nFolder:", root)

    for file in files:
        print("  ", file)



Folder: /content/dataset/FOLIO-main/data/v0.0
   folio-validation.jsonl
   folio-train.txt
   folio-train.jsonl
   folio-validation.txt


In [45]:
import json

train_file = os.path.join(DATA_PATH, "folio-train.jsonl")

with open(train_file, "r") as f:
    first_line = f.readline()

print(first_line)

{"story_id": 406, "example_id": 1131, "conclusion": "Rina is a person who jokes about being addicted to caffeine or unaware that caffeine is a drug.", "premises": ["All people who regularly drink coffee are dependent on caffeine.", "People either regularly drink coffee or joke about being addicted to caffeine.", "No one who jokes about being addicted to caffeine is unaware that caffeine is a drug.", "Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug.", "If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student. "], "premises-FOL": ["∀x (Drinks(x) → Dependent(x))", "∀x (Drinks(x) ⊕ Jokes(x))", "∀x (Jokes(x) → ¬Unaware(x))", "(Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina))", "¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rin

In [16]:
import json
import pandas as pd
import os

DATA_PATH = "/content/dataset/FOLIO-main/data/v0.0"

train_file = os.path.join(DATA_PATH, "folio-train.jsonl")

records = []

with open(train_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Number of examples:", len(records))


Number of examples: 1004


In [17]:
df = pd.DataFrame(records)

print(df.shape)
print(df.columns.tolist())


(1004, 7)
['story_id', 'example_id', 'conclusion', 'premises', 'premises-FOL', 'label', 'source']


In [18]:
df.head(2)


,story_id,example_id,conclusion,premises,premises-FOL,label,source
0,406,1131,Rina is a person who jokes about being addicte...,[All people who regularly drink coffee are dep...,"[∀x (Drinks(x) → Dependent(x)), ∀x (Drinks(x) ...",True,hyb
1,406,1132,Rina is either a person who jokes about being ...,[All people who regularly drink coffee are dep...,"[∀x (Drinks(x) → Dependent(x)), ∀x (Drinks(x) ...",True,hyb


In [19]:
print(df["label"].value_counts())


label
True       388
Unknown    330
False      286
Name: count, dtype: int64


In [20]:
foil_df = df[df["label"].isin(["True", "False"])].copy()

foil_df["target"] = foil_df["label"].map({
    "True": 1,
    "False": 0
})

print(foil_df["target"].value_counts())


target
1    388
0    286
Name: count, dtype: int64


In [21]:
foil_df[[
    "example_id",
    "conclusion",
    "label"
]].head(10)


,example_id,conclusion,label
0,1131,Rina is a person who jokes about being addicte...,True
1,1132,Rina is either a person who jokes about being ...,True
2,1133,Rina is either a person who regularly drinks c...,False
3,1134,If Rina is either a person who jokes about bei...,True
5,21,A Czech person wrote a book in 1946.,True
6,22,No choral conductor specialized in the perform...,False
8,1342,Sea eel is a paper.,False
9,1343,Sea eel breathes or is a paper.,True
10,393,A five-story building is built in 1915.,True
11,394,The Blake McFall Company Building is located i...,True


In [22]:
import re

def extract_predicates(fol_text):
    if not isinstance(fol_text, str):
        return []

    # Finds predicates such as Drinks(x), Dependent(x), Jokes(x)
    predicates = re.findall(r'\b([A-Z][A-Za-z0-9_]*)\s*\(', fol_text)

    return list(set(predicates))


In [23]:
example = records[0]["premises-FOL"]

print(example)

for statement in example:
    print(extract_predicates(statement))


['∀x (Drinks(x) → Dependent(x))', '∀x (Drinks(x) ⊕ Jokes(x))', '∀x (Jokes(x) → ¬Unaware(x))', '(Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina))', '¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina))']
['Drinks', 'Dependent']
['Jokes', 'Drinks']
['Unaware', 'Jokes']
['Unaware', 'Student']
['Dependent', 'Student']


In [24]:
all_predicates = set()

for _, row in foil_df.iterrows():

    # Conclusion
    all_predicates.update(
        extract_predicates(row["conclusion"])
    )

    # Premises
    for premise in row["premises-FOL"]:
        all_predicates.update(
            extract_predicates(premise)
        )

print("Number of predicates:", len(all_predicates))

print(sorted(all_predicates))


Number of predicates: 1504
['A1080p', 'AMC', 'AOC', 'ATypeOfCancer', 'About', 'AboutFuture', 'AboutLifeExperience', 'AboutTechnology', 'Accessory', 'Acclaimed', 'AcquiringData', 'Act', 'Actor', 'Actress', 'Adapt', 'AdjacentWall', 'Administers', 'AdministrativeCenterOf', 'Adult', 'AdventureFilm', 'Advocate', 'AgingAnalogy', 'AinderbyQuernhow', 'AirTight', 'Airforce', 'Album', 'AlbumAward', 'AlbumByBand', 'AlbumsReleased', 'AlfonsoLive', 'AlignHighSchool', 'Alive', 'Amateur', 'Amazon', 'AmbiortusDementjevi', 'Ambitious', 'American', 'AmericanAirlinesAircraft', 'AmericanComputerScientist', 'AmericanPolitician', 'Angry', 'Animal', 'AnimeHoldingCompany', 'Announce', 'Apartment', 'App', 'Appear', 'Apple', 'AppleJuice', 'ApplePay', 'ApplyHeat', 'ApplyVisa', 'Archeologist', 'Architect', 'Army', 'ArtPiece', 'ArtificialSatellite', 'ArtilleryFortification', 'Artist', 'Asian', 'AspiringArchitectureStudent', 'AssociatedWith', 'AtLuisParty', 'Athlete', 'Attend', 'Attended', 'AttendedSchoolWhereFrom'

In [25]:
example = records[0]

print("CONCLUSION:")
print(example["conclusion"])

print("\nPREMISES:")
for p in example["premises"]:
    print("-", p)

print("\nFOL PREMISES:")
for p in example["premises-FOL"]:
    print("-", p)

print("\nLABEL:")
print(example["label"])


CONCLUSION:
Rina is a person who jokes about being addicted to caffeine or unaware that caffeine is a drug.

PREMISES:
- All people who regularly drink coffee are dependent on caffeine.
- People either regularly drink coffee or joke about being addicted to caffeine.
- No one who jokes about being addicted to caffeine is unaware that caffeine is a drug.
- Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug.
- If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student. 

FOL PREMISES:
- ∀x (Drinks(x) → Dependent(x))
- ∀x (Drinks(x) ⊕ Jokes(x))
- ∀x (Jokes(x) → ¬Unaware(x))
- (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina))
- ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina))

LABEL:
True


In [26]:
print(records[0].keys())


dict_keys(['story_id', 'example_id', 'conclusion', 'premises', 'premises-FOL', 'label', 'source'])


In [27]:
def parse_simple_rule(fol):
    pattern = r'∀x\s*\(\s*([A-Z][A-Za-z0-9_]*)\(x\)\s*→\s*([¬A-Z][A-Za-z0-9_]*)\(x\)\s*\)'

    match = re.search(pattern, fol)

    if match:
        body = match.group(1)
        head = match.group(2)

        return head, body

    return None


In [28]:
test = "∀x (Drinks(x) → Dependent(x))"

print(parse_simple_rule(test))


('Dependent', 'Drinks')


In [29]:
rules = []

for record in records:

    for premise in record["premises-FOL"]:

        result = parse_simple_rule(premise)

        if result:
            head, body = result

            rules.append({
                "head": head,
                "body": body
            })

rules_df = pd.DataFrame(rules)

print(rules_df.head(20))


                head             body
0          Dependent           Drinks
1           ¬Unaware            Jokes
2          Dependent           Drinks
3           ¬Unaware            Jokes
4          Dependent           Drinks
5           ¬Unaware            Jokes
6          Dependent           Drinks
7           ¬Unaware            Jokes
8           Musician  ChoralConductor
9           Musician  ChoralConductor
10          Musician  ChoralConductor
11          Building            Blake
12          Building            Blake
13          Building            Blake
14       ThreeMovies              AMC
15  ¬WatchTVInCinema   PreferTVSeries
16    PreferTVSeries              HBO
17       ThreeMovies              AMC
18  ¬WatchTVInCinema   PreferTVSeries
19    PreferTVSeries              HBO


In [30]:
print("Number of simple rules:", len(rules_df))



Number of simple rules: 1948


In [31]:
rules_df.value_counts()


,,count
head,body,
ProfessionalBasketballPlayer,NBAPlayer,15
SoccerPlayer,Defender,12
¬MansionHouse,UrbanArea,10
¬PlaysLots,BadChess,10
¬ProvedToBeFalse,Fact,10
...,...,...
HasFur,Rabbit,1
Electronic,IPhone,1
ExportFall,Embargo,1


In [32]:
print(records[0].keys())
print(df["label"].value_counts())
print(rules_df.head(20))
print("Number of simple rules:", len(rules_df))


dict_keys(['story_id', 'example_id', 'conclusion', 'premises', 'premises-FOL', 'label', 'source'])
label
True       388
Unknown    330
False      286
Name: count, dtype: int64
                head             body
0          Dependent           Drinks
1           ¬Unaware            Jokes
2          Dependent           Drinks
3           ¬Unaware            Jokes
4          Dependent           Drinks
5           ¬Unaware            Jokes
6          Dependent           Drinks
7           ¬Unaware            Jokes
8           Musician  ChoralConductor
9           Musician  ChoralConductor
10          Musician  ChoralConductor
11          Building            Blake
12          Building            Blake
13          Building            Blake
14       ThreeMovies              AMC
15  ¬WatchTVInCinema   PreferTVSeries
16    PreferTVSeries              HBO
17       ThreeMovies              AMC
18  ¬WatchTVInCinema   PreferTVSeries
19    PreferTVSeries              HBO
Number of simple rules: 19

In [33]:
import re

def parse_simple_rule(fol):
    """
    Parse rules of the form:
    ∀x (Predicate1(x) → Predicate2(x))
    """

    pattern = (
        r'∀x\s*\(\s*'
        r'([A-Z][A-Za-z0-9_]*)\(x\)'
        r'\s*→\s*'
        r'(¬?)([A-Z][A-Za-z0-9_]*)\(x\)'
        r'\s*\)'
    )

    match = re.search(pattern, fol)

    if match:
        body = match.group(1)
        negation = match.group(2)
        head = match.group(3)

        if negation:
            head = "NOT_" + head

        return head, body

    return None


In [34]:
tests = [
    "∀x (Drinks(x) → Dependent(x))",
    "∀x (Jokes(x) → ¬Unaware(x))"
]

for t in tests:
    print(t)
    print(parse_simple_rule(t))
    print()


∀x (Drinks(x) → Dependent(x))
('Dependent', 'Drinks')

∀x (Jokes(x) → ¬Unaware(x))
('NOT_Unaware', 'Jokes')



In [35]:
rules = []

for record in records:

    for premise in record["premises-FOL"]:

        result = parse_simple_rule(premise)

        if result:
            head, body = result

            rules.append({
                "head": head,
                "body": body
            })

rules_df = pd.DataFrame(rules)

print("Number of simple rules:", len(rules_df))
rules_df.head(20)


Number of simple rules: 1948


,head,body
0,Dependent,Drinks
1,NOT_Unaware,Jokes
2,Dependent,Drinks
3,NOT_Unaware,Jokes
4,Dependent,Drinks
5,NOT_Unaware,Jokes
6,Dependent,Drinks
7,NOT_Unaware,Jokes
8,Musician,ChoralConductor
9,Musician,ChoralConductor


In [36]:
facts = {
    "Drinks": {"alice", "bob", "charlie"},
    "Jokes": {"david", "emma"},
    "Student": {"alice", "david"},
    "Unaware": {"alice", "bob"}
}

print(facts)


{'Drinks': {'bob', 'charlie', 'alice'}, 'Jokes': {'emma', 'david'}, 'Student': {'alice', 'david'}, 'Unaware': {'bob', 'alice'}}


In [37]:
all_people = {
    "alice",
    "bob",
    "charlie",
    "david",
    "emma",
    "frank"
}

positive_examples = [
    x for x in all_people
    if x in facts["Drinks"]
]

negative_examples = [
    x for x in all_people
    if x not in facts["Drinks"]
]

print("Positive:", positive_examples)
print("Negative:", negative_examples)


Positive: ['bob', 'charlie', 'alice']
Negative: ['frank', 'emma', 'david']


In [38]:
def foil_learn(target,
               positive_examples,
               negative_examples,
               facts):

    learned_rules = []

    remaining_positive = set(positive_examples)

    while remaining_positive:

        rule_body = []

        current_positive = set(remaining_positive)
        current_negative = set(negative_examples)

        # Find literals that eliminate negative examples
        while current_negative:

            best_literal = None
            best_score = -1

            for predicate in facts:

                covered_positive = (
                    current_positive & facts[predicate]
                )

                covered_negative = (
                    current_negative & facts[predicate]
                )

                if len(covered_negative) == 0:

                    score = len(covered_positive)

                    if score > best_score:
                        best_score = score
                        best_literal = predicate

            if best_literal is None:
                break

            rule_body.append(best_literal)

            current_positive &= facts[best_literal]
            current_negative &= facts[best_literal]

        if rule_body:

            learned_rules.append({
                "target": target,
                "body": rule_body
            })

            remaining_positive -= current_positive

        else:
            break

    return learned_rules


In [39]:
rules = foil_learn(
    target="Dependent",
    positive_examples=positive_examples,
    negative_examples=negative_examples,
    facts=facts
)

rules


[{'target': 'Dependent', 'body': ['Drinks']}]

In [40]:
def print_rules(rules):

    for rule in rules:

        body = ", ".join(
            f"{p}(X)"
            for p in rule["body"]
        )

        print(
            f"{rule['target']}(X) :- {body}"
        )


print_rules(rules)


Dependent(X) :- Drinks(X)


In [42]:
import math

def foil_gain(p, n, P, N):
    """
    Simplified FOIL information gain.

    p = positive examples covered after adding literal
    n = negative examples covered after adding literal

    P = positive examples before
    N = negative examples before
    """

    if p == 0:
        return -float("inf")

    if p + n == 0:
        return -float("inf")

    before = math.log2(
        P / (P + N)
    )

    after = math.log2(
        p / (p + n)
    )

    return p * (after - before)


In [43]:
gain = foil_gain(
    p=3,
    n=0,
    P=3,
    N=3
)

print("Information gain:", gain)


Information gain: 3.0


## Step 23 — Create a proper relational dataset

Before connecting the full FOLIO dataset, let's create a small relational representation. This lets us verify that our FOIL implementation works.

Run:

```python
# Knowledge base

facts = [
    ("parent", "alice", "bob"),
    ("parent", "bob", "charlie"),
    ("parent", "alice", "david"),
    ("parent", "david", "emma"),
    ("parent", "charlie", "frank"),
    ("parent", "emma", "george"),

    ("female", "alice"),
    ("female", "emma"),

    ("male", "bob"),
    ("male", "charlie"),
    ("male", "david"),
    ("male", "frank"),
]
```

We want FOIL to learn:

```
grandparent(X,Y) :-
    parent(X,Z),
    parent(Z,Y)
```


In [46]:
# Knowledge base

facts = [
    ("parent", "alice", "bob"),
    ("parent", "bob", "charlie"),
    ("parent", "alice", "david"),
    ("parent", "david", "emma"),
    ("parent", "charlie", "frank"),
    ("parent", "emma", "george"),

    ("female", "alice"),
    ("female", "emma"),

    ("male", "bob"),
    ("male", "charlie"),
    ("male", "david"),
    ("male", "frank"),
]

---

 ## Step 24 — Define examples

 Positive examples:

```python
positive_examples = [
    ("alice", "charlie"),
    ("alice", "emma"),
    ("bob", "frank"),
    ("alice", "george")
]
```

 Negative examples:

```python
negative_examples = [
    ("alice", "bob"),
    ("bob", "alice"),
    ("charlie", "alice"),
    ("david", "bob"),
    ("alice", "frank")
]
```

 Target:

```python
target = "grandparent"
```


In [47]:
positive_examples = [
    ("alice", "charlie"),
    ("alice", "emma"),
    ("bob", "frank"),
    ("alice", "george")
]

negative_examples = [
    ("alice", "bob"),
    ("bob", "alice"),
    ("charlie", "alice"),
    ("david", "bob"),
    ("alice", "frank")
]

target = "grandparent"

---

 ## Step 25 — Convert facts into a convenient structure

```python
from collections import defaultdict

knowledge_base = defaultdict(set)

for relation, *args in facts:
    knowledge_base[relation].add(tuple(args))

knowledge_base
```


In [48]:
from collections import defaultdict

knowledge_base = defaultdict(set)

for relation, *args in facts:
    knowledge_base[relation].add(tuple(args))

knowledge_base

defaultdict(set,
            {'parent': {('alice', 'bob'),
              ('alice', 'david'),
              ('bob', 'charlie'),
              ('charlie', 'frank'),
              ('david', 'emma'),
              ('emma', 'george')},
             'female': {('alice',), ('emma',)},
             'male': {('bob',), ('charlie',), ('david',), ('frank',)}})

---

 # Step 26 — Implement variable matching

 This is the important part.

 A literal could be:

```
parent(X,Z)
```

 and an example might be:

```
X = alice
Y = charlie
```

 FOIL needs to find a `Z`.

 For:

```
parent(alice,Z)
```

 the database contains:

```
parent(alice,bob)
```

 so:

```
Z = bob
```

 Implement:

```python
def match_literal(literal, substitution, knowledge_base):
    """
    literal example:
        ("parent", "X", "Z")

    substitution:
        {"X": "alice", "Y": "charlie"}

    returns all valid substitutions.
    """

    relation, *args_literal = literal

    results = []

    if relation not in knowledge_base:
        return []

    for fact_tuple in knowledge_base[relation]:
        new_sub = substitution.copy()
        match = True

        if len(args_literal) != len(fact_tuple):
            continue

        for i, arg_literal in enumerate(args_literal):
            fact_val = fact_tuple[i]

            if arg_literal.isupper(): # It's a variable
                if arg_literal in new_sub:
                    if new_sub[arg_literal] != fact_val:
                        match = False
                        break
                else:
                    new_sub[arg_literal] = fact_val
            elif arg_literal != fact_val: # It's a constant and doesn't match
                match = False
                break
        
        if match:
            results.append(new_sub)

    return results
```


In [49]:
def match_literal(literal, substitution, knowledge_base):
    """
    literal example:
        ("parent", "X", "Z")

    substitution:
        {"X": "alice", "Y": "charlie"}

    returns all valid substitutions.
    """

    relation, *args_literal = literal

    results = []

    if relation not in knowledge_base:
        return []

    for fact_tuple in knowledge_base[relation]:
        new_sub = substitution.copy()
        match = True

        if len(args_literal) != len(fact_tuple):
            continue

        for i, arg_literal in enumerate(args_literal):
            fact_val = fact_tuple[i]

            if arg_literal.isupper(): # It's a variable
                if arg_literal in new_sub:
                    if new_sub[arg_literal] != fact_val:
                        match = False
                        break
                else:
                    new_sub[arg_literal] = fact_val
            elif arg_literal != fact_val: # It's a constant and doesn't match
                match = False
                break

        if match:
            results.append(new_sub)

    return results

---

 # Step 27 — Test matching

 Run:

```python
literal = ("parent", "X", "Z")

substitution = {
    "X": "alice"
}

matches = match_literal(
    literal,
    substitution,
    knowledge_base
)

print(matches)
```

 You should see something similar to:

```
[
    {'X': 'alice', 'Z': 'bob'},
    {'X': 'alice', 'Z': 'david'}
]
```

 This means:

```
parent(alice,Z)
```

 can produce:

```
Z = bob
Z = david
```

 Excellent.


In [50]:
literal = ("parent", "X", "Z")

substitution = {
    "X": "alice"
}

matches = match_literal(
    literal,
    substitution,
    knowledge_base
)

print(matches)

[{'X': 'alice', 'Z': 'david'}, {'X': 'alice', 'Z': 'bob'}]


---

 # Step 28 — Evaluate multiple literals

 Now suppose our rule body is:

```
parent(X,Z)
parent(Z,Y)
```

 We need to execute them sequentially.

```python
def evaluate_body(body, initial_substitution, knowledge_base):

    substitutions = [initial_substitution]

    for literal in body:

        new_substitutions = []

        for sub in substitutions:

            matches = match_literal(
                literal,
                sub,
                knowledge_base
            )

            new_substitutions.extend(matches)

        substitutions = new_substitutions

        if not substitutions:
            break

    return substitutions
```


In [51]:
def evaluate_body(body, initial_substitution, knowledge_base):

    substitutions = [initial_substitution]

    for literal in body:

        new_substitutions = []

        for sub in substitutions:

            matches = match_literal(
                literal,
                sub,
                knowledge_base
            )

            new_substitutions.extend(matches)

        substitutions = new_substitutions

        if not substitutions:
            break

    return substitutions

---

 # Step 29 — Test a two-literal rule

```python
body = [
    ("parent", "X", "Z"),
    ("parent", "Z", "Y")
]

result = evaluate_body(
    body,
    {"X": "alice", "Y": "charlie"},
    knowledge_base
)

print(result)
```

 Expected:

```
[{'X': 'alice', 'Y': 'charlie', 'Z': 'bob'}]
```

 That's huge.

 Our system has just proven:

```
parent(alice,bob)
parent(bob,charlie)
```

 therefore:

```
grandparent(alice,charlie)
```


In [52]:
body = [
    ("parent", "X", "Z"),
    ("parent", "Z", "Y")
]

result = evaluate_body(
    body,
    {"X": "alice", "Y": "charlie"},
    knowledge_base
)

print(result)

[{'X': 'alice', 'Y': 'charlie', 'Z': 'bob'}]


---

 # Step 30 — Check whether a rule covers an example

```python
def covers(rule_body, example, knowledge_base):

    substitution = {
        "X": example[0],
        "Y": example[1]
    }

    results = evaluate_body(
        rule_body,
        substitution,
        knowledge_base
    )

    return len(results) > 0
```

 Test:

```python
rule_body = [
    ("parent", "X", "Z"),
    ("parent", "Z", "Y")
]

print(
    covers(
        rule_body,
        ("alice", "charlie"),
        knowledge_base
    )
)
```

 Output:

```
True
```

 Test a negative:

```python
print(
    covers(
        rule_body,
        ("alice", "bob"),
        knowledge_base
    )
)
```

 Output:

```
False
```

 So our rule correctly separates those two examples.


In [53]:
def covers(rule_body, example, knowledge_base):

    substitution = {
        "X": example[0],
        "Y": example[1]
    }

    results = evaluate_body(
        rule_body,
        substitution,
        knowledge_base
    )

    return len(results) > 0

In [54]:
rule_body = [
    ("parent", "X", "Z"),
    ("parent", "Z", "Y")
]

print(
    covers(
        rule_body,
        ("alice", "charlie"),
        knowledge_base
    )
)

print(
    covers(
        rule_body,
        ("alice", "bob"),
        knowledge_base
    )
)

True
False


---

 # Step 31 — Generate candidate literals

 FOIL needs to try different possibilities.

 We'll allow:

```
parent(X,Z)
parent(Z,Y)
parent(X,Y)
female(X)
female(Y)
male(X)
male(Y)
```

 Implement:

```python
def generate_candidates():

    candidates = []

    relations = ["parent", "female", "male"]

    variables = ["X", "Y", "Z"]

    for relation in relations:

        # Binary relations
        if relation == "parent":

            for a in variables:
                for b in variables:

                    # Ensure distinct variables or that X is not Z/Y and Y is not Z/X etc.
                    if a == b: # e.g., parent(X,X) is not useful for grandparent
                        continue

                    candidates.append(
                        (relation, a, b)
                    )

        # Unary relations
        else:

            for variable in ["X", "Y", "Z"]:

                candidates.append(
                    (relation, variable)
                )

    return candidates
```

 Run:

```python
candidates = generate_candidates()

for c in candidates:
    print(c)
```


In [55]:
def generate_candidates():

    candidates = []

    relations = ["parent", "female", "male"]

    variables = ["X", "Y", "Z"]

    for relation in relations:

        # Binary relations
        if relation == "parent":

            for a in variables:
                for b in variables:

                    if a == b: # e.g., parent(X,X) is not useful for grandparent
                        continue

                    candidates.append(
                        (relation, a, b)
                    )

        # Unary relations
        else:

            for variable in ["X", "Y", "Z"]:

                candidates.append(
                    (relation, variable)
                )

    return candidates

candidates = generate_candidates()

for c in candidates:
    print(c)

('parent', 'X', 'Y')
('parent', 'X', 'Z')
('parent', 'Y', 'X')
('parent', 'Y', 'Z')
('parent', 'Z', 'X')
('parent', 'Z', 'Y')
('female', 'X')
('female', 'Y')
('female', 'Z')
('male', 'X')
('male', 'Y')
('male', 'Z')


---

 # Step 32 — Calculate FOIL information gain

 Now we use the actual idea behind FOIL.

```python
import math

def information_gain(P, N, p, n):

    if p == 0:
        return -float("inf")

    if p + n == 0:
        return -float("inf")

    before = P / (P + N)

    after = p / (p + n)

    if before == 0 or after == 0:
        return -float("inf")

    return p * (
        math.log2(after) -
        math.log2(before)
    )
```


In [56]:
import math

def information_gain(P, N, p, n):

    if p == 0:
        return -float("inf")

    if p + n == 0:
        return -float("inf")

    before = P / (P + N)

    after = p / (p + n)

    if before == 0 or after == 0:
        return -float("inf")

    return p * (
        math.log2(after) -
        math.log2(before)
    )

---

 # Step 33 — Evaluate candidate literals

```python
def evaluate_candidate(
    literal,
    positive,
    negative,
    knowledge_base
):

    covered_positive = []

    covered_negative = []

    for example in positive:

        if covers(
            [literal],
            example,
            knowledge_base
        ):
            covered_positive.append(example)

    for example in negative:

        if covers(
            [literal],
            example,
            knowledge_base
        ):
            covered_negative.append(example)

    return (
        covered_positive,
        covered_negative
    )
```


In [57]:
def evaluate_candidate(
    literal,
    positive,
    negative,
    knowledge_base
):

    covered_positive = []

    covered_negative = []

    for example in positive:

        if covers(
            [literal],
            example,
            knowledge_base
        ):
            covered_positive.append(example)

    for example in negative:

        if covers(
            [literal],
            example,
            knowledge_base
        ):
            covered_negative.append(example)

    return (
        covered_positive,
        covered_negative
    )

---

 # Step 34 — Find the best first literal

```python
best_literal = None
best_gain = -float("inf")

P = len(positive_examples)
N = len(negative_examples)

for literal in candidates:

    pos, neg = evaluate_candidate(
        literal,
        positive_examples,
        negative_examples,
        knowledge_base
    )

    gain = information_gain(
        P,
        N,
        len(pos),
        len(neg)
    )

    print(
        literal,
        "positive =", len(pos),
        "negative =", len(neg),
        "gain =", round(gain, 3)
    )

    if gain > best_gain:

        best_gain = gain
        best_literal = literal

print("\nBEST:")
print(best_literal)
print("Gain:", best_gain)
```

 This is where FOIL starts making the decision **automatically**.


In [58]:
best_literal = None
best_gain = -float("inf")

P = len(positive_examples)
N = len(negative_examples)

for literal in candidates:

    pos, neg = evaluate_candidate(
        literal,
        positive_examples,
        negative_examples,
        knowledge_base
    )

    gain = information_gain(
        P,
        N,
        len(pos),
        len(neg)
    )

    print(
        literal,
        "positive =", len(pos),
        "negative =", len(neg),
        "gain =", round(gain, 3)
    )

    if gain > best_gain:

        best_gain = gain
        best_literal = literal

print("\nBEST:")
print(best_literal)
print("Gain:", best_gain)

('parent', 'X', 'Y') positive = 0 negative = 1 gain = -inf
('parent', 'X', 'Z') positive = 4 negative = 5 gain = 0.0
('parent', 'Y', 'X') positive = 0 negative = 1 gain = -inf
('parent', 'Y', 'Z') positive = 2 negative = 4 gain = -0.83
('parent', 'Z', 'X') positive = 1 negative = 3 gain = -0.83
('parent', 'Z', 'Y') positive = 4 negative = 3 gain = 1.45
('female', 'X') positive = 3 negative = 2 gain = 1.299
('female', 'Y') positive = 1 negative = 2 gain = -0.415
('female', 'Z') positive = 4 negative = 5 gain = 0.0
('male', 'X') positive = 1 negative = 3 gain = -0.83
('male', 'Y') positive = 2 negative = 3 gain = -0.304
('male', 'Z') positive = 4 negative = 5 gain = 0.0

BEST:
('parent', 'Z', 'Y')
Gain: 1.4502803175388328


---

 # Step 35 — Build the FOIL learner

 Now combine everything.

```python
def foil(
    positive_examples,
    negative_examples,
    knowledge_base
):

    rule_body = []

    remaining_negative = negative_examples.copy()

    while remaining_negative:

        candidates = generate_candidates()

        best_literal = None
        best_gain = -float("inf")

        P = len(positive_examples)
        N = len(remaining_negative)

        for literal in candidates:

            candidate_body = rule_body + [literal]

            covered_pos = [
                e for e in positive_examples
                if covers(
                    candidate_body,
                    e,
                    knowledge_base
                )
            ]

            covered_neg = [
                e for e in remaining_negative
                if covers(
                    candidate_body,
                    e,
                    knowledge_base
                )
            ]

            gain = information_gain(
                P,
                N,
                len(covered_pos),
                len(covered_neg)
            )

            if gain > best_gain:

                best_gain = gain
                best_literal = literal

        if best_literal is None:
            break

        rule_body.append(best_literal)

        remaining_negative = [
            e for e in remaining_negative
            if covers(
                rule_body,
                e,
                knowledge_base
            )
        ]

        print(
            "Added:",
            best_literal,
            "| Remaining negatives:",
            len(remaining_negative)
        )

        if len(rule_body) > 5:
            break

    return rule_body
```


In [59]:
def foil(
    positive_examples,
    negative_examples,
    knowledge_base
):

    rule_body = []

    remaining_negative = negative_examples.copy()

    while remaining_negative:

        candidates = generate_candidates()

        best_literal = None
        best_gain = -float("inf")

        P = len(positive_examples)
        N = len(remaining_negative)

        for literal in candidates:

            candidate_body = rule_body + [literal]

            covered_pos = [
                e for e in positive_examples
                if covers(
                    candidate_body,
                    e,
                    knowledge_base
                )
            ]

            covered_neg = [
                e for e in remaining_negative
                if covers(
                    candidate_body,
                    e,
                    knowledge_base
                )
            ]

            gain = information_gain(
                P,
                N,
                len(covered_pos),
                len(covered_neg)
            )

            if gain > best_gain:

                best_gain = gain
                best_literal = literal

        if best_literal is None:
            break

        rule_body.append(best_literal)

        remaining_negative = [
            e for e in remaining_negative
            if covers(
                rule_body,
                e,
                knowledge_base
            )
        ]

        print(
            "Added:",
            best_literal,
            "| Remaining negatives:",
            len(remaining_negative)
        )

        if len(rule_body) > 5:
            break

    return rule_body

---

 # Step 36 — Run FOIL 🎯

```python
learned_body = foil(
    positive_examples,
    negative_examples,
    knowledge_base
)

print("\nLearned rule:")
print(learned_body)
```

 We're aiming for something equivalent to:

```
parent(X,Z)
parent(Z,Y)
```

 which corresponds to:

```
grandparent(X,Y) :-
    parent(X,Z),
    parent(Z,Y)
```


In [60]:
learned_body = foil(
    positive_examples,
    negative_examples,
    knowledge_base
)

print("\nLearned rule:")
print(learned_body)

Added: ('parent', 'Z', 'Y') | Remaining negatives: 3
Added: ('parent', 'X', 'Z') | Remaining negatives: 0

Learned rule:
[('parent', 'Z', 'Y'), ('parent', 'X', 'Z')]


---

 # Step 37 — Pretty-print it

```python
def print_rule(target, body):

    body_text = ", ".join(
        f"{x[0]}({', '.join(x[1:])})"
        for x in body
    )

    print(
        f"{target}(X,Y) :- {body_text}"
    )

print_rule(
    "grandparent",
    learned_body
)
```


In [61]:
def print_rule(target, body):

    body_text = ", ".join(
        f"{x[0]}({', '.join(x[1:])})"
        for x in body
    )

    print(
        f"{target}(X,Y) :- {body_text}"
    )

print_rule(
    "grandparent",
    learned_body
)

grandparent(X,Y) :- parent(Z, Y), parent(X, Z)


## Step 40 — Define Target Predicate and Generate Positive/Negative Examples from FOLIO

To apply our FOIL algorithm, we need to choose a specific target predicate we want to learn rules for. Let's pick a unary predicate, `Student`, as an example. We will then generate a list of positive and negative examples for this predicate based on the `folio_knowledge_base` and the `all_individuals` we've collected. This will effectively create the training data for FOIL.

```python
target_predicate_name = "Student"

folio_positive_examples = []
folio_negative_examples = []

# Ensure the target predicate exists in the knowledge base, even if empty
# to avoid KeyError when accessing folio_knowledge_base[target_predicate_name]
if target_predicate_name not in folio_knowledge_base:
    folio_knowledge_base[target_predicate_name] = set()

for individual in all_individuals:
    # Facts are stored as tuples, so for unary, it's (individual,)
    if (individual,) in folio_knowledge_base[target_predicate_name]:
        folio_positive_examples.append(individual)
    else:
        # An individual is a negative example if it's not explicitly in the positive facts
        folio_negative_examples.append(individual)

print(f"Target Predicate: {target_predicate_name}")
print(f"Number of Positive Examples for {target_predicate_name}: {len(folio_positive_examples)}")
print(f"Positive Examples (first 5): {folio_positive_examples[:5]}")
print(f"Number of Negative Examples for {target_predicate_name}: {len(folio_negative_examples)}")
print(f"Negative Examples (first 5): {folio_negative_examples[:5]}")


In [67]:
print(f"DEBUG: all_individuals has {len(all_individuals)} items.")

target_predicate_name = "Dependent" # Changed from "Student" for debugging

folio_positive_examples = []
folio_negative_examples = []

# Ensure the target predicate exists in the knowledge base, even if empty
# to avoid KeyError when accessing folio_knowledge_base[target_predicate_name]
if target_predicate_name not in folio_knowledge_base:
    folio_knowledge_base[target_predicate_name] = set()

for individual in all_individuals:
    # Facts are stored as tuples, so for unary, it's (individual,)
    if (individual,) in folio_knowledge_base[target_predicate_name]:
        folio_positive_examples.append(individual)
    else:
        # An individual is a negative example if it's not explicitly in the positive facts
        folio_negative_examples.append(individual)

print(f"Target Predicate: {target_predicate_name}")
print(f"Number of Positive Examples for {target_predicate_name}: {len(folio_positive_examples)}")
print(f"Positive Examples (first 5): {folio_positive_examples[:5]}")
print(f"Number of Negative Examples for {target_predicate_name}: {len(folio_negative_examples)}")
print(f"Negative Examples (first 5): {folio_negative_examples[:5]}")

DEBUG: all_individuals has 611 items.
Target Predicate: Dependent
Number of Positive Examples for Dependent: 1
Positive Examples (first 5): ['rina']
Number of Negative Examples for Dependent: 610
Negative Examples (first 5): ['neocrepidoderacorpulenta', 'year2016', 'monhoff', 'abc', 'jesse']


## Step 41 — Adapt FOIL to FOLIO Dataset

Now that we have successfully parsed the FOLIO premises into `folio_knowledge_base` and generated `folio_positive_examples` and `folio_negative_examples` for our target predicate (`Dependent`), we need to adapt our FOIL algorithm to use these structures.

This involves:
1.  **Refining `is_variable`**: A helper function to identify variables within FOLIO arguments.
2.  **Adapting `folio_match_literal`**: This function will be similar to our previous `match_literal` but will operate on the `folio_knowledge_base` and account for different argument structures (e.g., unary predicates having `('item',)` tuples).
3.  **Adapting `folio_evaluate_body`**: Similar to the previous `evaluate_body`, but using `folio_match_literal`.
4.  **Adapting `folio_covers`**: Checks if a rule body covers an example, using `folio_evaluate_body`.
5.  **Adapting `folio_generate_candidates`**: This will generate candidate literals from `folio_predicate_signatures` by replacing arguments with `X` and `Z` variables, consistent with our FOIL implementation.
6.  **Adapting `folio_foil`**: The main FOIL algorithm, using the new `folio_covers`, `folio_generate_candidates`, and `information_gain` functions.

In [68]:
def is_variable(arg):
    """
    Determines if an argument string represents a variable (X, Y, Z).
    Our FOIL implementation uses uppercase letters for variables.
    """
    return arg.isupper() and len(arg) == 1

def folio_match_literal(literal, substitution, knowledge_base):
    """
    literal example: ('PredicateName', 'X', 'Z') or ('PredicateName', 'X')
    substitution: {'X': 'rina'}
    knowledge_base: {'Dependent': {('rina',)}}

    Returns all valid substitutions.
    """
    relation = literal[0]
    args_literal = literal[1:]

    results = []

    if relation not in knowledge_base:
        return []

    for fact_tuple in knowledge_base[relation]:
        new_sub = substitution.copy()
        match = True

        if len(args_literal) != len(fact_tuple):
            continue # Mismatch in arity

        for i, arg_literal_item in enumerate(args_literal):
            fact_val = fact_tuple[i]

            if is_variable(arg_literal_item):
                if arg_literal_item in new_sub:
                    if new_sub[arg_literal_item] != fact_val:
                        match = False
                        break
                else:
                    new_sub[arg_literal_item] = fact_val
            elif arg_literal_item != fact_val:
                match = False
                break

        if match:
            results.append(new_sub)

    return results

def folio_evaluate_body(body, initial_substitution, knowledge_base):
    """
    Evaluates a list of literals (rule body) sequentially.
    """
    substitutions = [initial_substitution]

    for literal in body:
        new_substitutions = []
        for sub in substitutions:
            matches = folio_match_literal(literal, sub, knowledge_base)
            new_substitutions.extend(matches)
        substitutions = new_substitutions

        if not substitutions:
            break

    return substitutions

def folio_covers(rule_body, example, knowledge_base, is_unary_target=True):
    """
    Checks if a rule body covers an example for a unary or binary target predicate.
    """
    if is_unary_target:
        # For unary predicates, the example is just the individual (e.g., 'rina')
        # We'll map it to 'X' for consistency with rule variable naming.
        initial_substitution = {"X": example}
    else:
        # For binary predicates, example is (arg1, arg2)
        initial_substitution = {"X": example[0], "Y": example[1]}

    results = folio_evaluate_body(rule_body, initial_substitution, knowledge_base)
    return len(results) > 0

def folio_generate_candidates(predicate_signatures):
    """
    Generates candidate literals based on extracted FOLIO predicate signatures.
    Unary predicates will be `(PredicateName, 'X')`, `(PredicateName, 'Z')`.
    Binary predicates will be `(PredicateName, 'X', 'Y')`, `(PredicateName, 'X', 'Z')` etc.
    We avoid `(PredicateName, 'X', 'X')` etc. for now, assuming distinct variables where applicable.
    """
    candidates = []
    variables = ['X', 'Y', 'Z'] # Common variables used in FOIL

    for pred_name, arity in predicate_signatures:
        if arity == 1:
            for var in variables:
                candidates.append((pred_name, var))
        elif arity == 2:
            for var1 in variables:
                for var2 in variables:
                    if var1 != var2: # Avoid literals like P(X,X) unless specifically needed
                        candidates.append((pred_name, var1, var2))
        # Add more arities if necessary

    return candidates

def folio_foil(
    positive_examples,
    negative_examples,
    knowledge_base,
    predicate_signatures,
    is_unary_target=True
):
    """
    Adapted FOIL algorithm for FOLIO data.
    """
    learned_rules = []
    # Convert to sets for efficient removal
    remaining_positive = set(positive_examples)
    all_negative = set(negative_examples) # All negative examples for the target predicate

    iteration = 0
    MAX_ITERATIONS = 10 # Prevent infinite loops

    print("\n--- Starting FOLIO FOIL learning ---")

    while remaining_positive and iteration < MAX_ITERATIONS:
        iteration += 1
        print(f"\nIteration {iteration}: Learning a new rule. Remaining positive: {len(remaining_positive)}")

        rule_body = []
        current_negative_covered_by_rule = set(all_negative) # Negative examples that *could* still be covered

        # Find literals to add to the current rule
        while True:
            candidates = folio_generate_candidates(predicate_signatures)

            best_literal = None
            best_gain = -float('inf')

            # P and N here refer to the examples for the *current* rule's head
            # which means remaining positive examples and all negative examples for the target.
            P_total = len(remaining_positive)
            N_total = len(all_negative)

            if P_total == 0: # No more positive examples to cover
                break

            print(f"  Considering literals for current rule (body: {rule_body})")

            for literal in candidates:
                # Avoid adding the same literal twice or adding a literal that creates redundancy
                # This is a simplification; a full FOIL might handle variable renaming
                if literal in rule_body:
                    continue

                candidate_body = rule_body + [literal]

                # Count how many of the *remaining* positive examples are covered
                covered_pos = [
                    e for e in remaining_positive
                    if folio_covers(candidate_body, e, knowledge_base, is_unary_target)
                ]

                # Count how many of the *total* negative examples are covered
                covered_neg = [
                    e for e in all_negative
                    if folio_covers(candidate_body, e, knowledge_base, is_unary_target)
                ]

                gain = information_gain(
                    P_total,
                    N_total,
                    len(covered_pos),
                    len(covered_neg)
                )

                # print(f"    Candidate {literal}: pos={len(covered_pos)}, neg={len(covered_neg)}, gain={gain:.3f}")

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            # If no literal improves gain significantly or no literal was found
            if best_literal is None or best_gain <= 0: # Only add if there's actual gain
                break

            rule_body.append(best_literal)
            print(f"  Added literal: {best_literal}, Gain: {best_gain:.3f}")

            # Update which negative examples are still covered by the *growing* rule
            # We want to eliminate all negative examples. If the rule now covers negatives,
            # those are the ones we need to address with further literals.
            current_negative_covered_by_rule = set([
                e for e in all_negative
                if folio_covers(rule_body, e, knowledge_base, is_unary_target)
            ])

            if not current_negative_covered_by_rule: # Rule successfully covers no negative examples
                break

            # Optional: if the rule body gets too long, stop adding literals to it
            if len(rule_body) > 3: # Limit complexity of learned rules
                print("  Rule body too long, breaking...")
                break

        # After building one rule, add it to the learned rules
        if rule_body:
            learned_rules.append(rule_body)
            print(f"Learned rule: {rule_body}")

            # Remove positive examples covered by this new rule
            covered_by_new_rule = set([
                e for e in remaining_positive
                if folio_covers(rule_body, e, knowledge_base, is_unary_target)
            ])
            remaining_positive -= covered_by_new_rule
            print(f"  Covered {len(covered_by_new_rule)} positive examples. {len(remaining_positive)} remaining.")
        else:
            # If no rule could be learned (e.g., no literal provided gain),
            # break to prevent infinite loop for unlearnable examples.
            print("No rule learned in this iteration, breaking.")
            break

    return learned_rules


def print_folio_rule(target_predicate, rule_body, is_unary_target=True):
    """
    Pretty-prints a learned rule for FOLIO data.
    """
    head_vars = "(X)" if is_unary_target else "(X,Y)"
    body_literals = []
    for literal in rule_body:
        pred_name = literal[0]
        args = ", ".join(literal[1:])
        body_literals.append(f"{pred_name}({args})")

    body_text = ", ".join(body_literals)
    if not body_text:
        body_text = "TRUE"

    print(f"{target_predicate}{head_vars} :- {body_text}")

# --- Run FOIL on FOLIO data ---
print(f"\nTarget Predicate: {target_predicate_name}")
print(f"Positive Examples: {len(folio_positive_examples)}")
print(f"Negative Examples: {len(folio_negative_examples)}")

folio_learned_rules = folio_foil(
    folio_positive_examples,
    folio_negative_examples,
    folio_knowledge_base,
    folio_predicate_signatures,
    is_unary_target=True
)

print("\n==============================")
print("FOLIO LEARNED RULES")
print("==============================")
for rule_body in folio_learned_rules:
    print_folio_rule(target_predicate_name, rule_body, is_unary_target=True)


Target Predicate: Dependent
Positive Examples: 1
Negative Examples: 610

--- Starting FOLIO FOIL learning ---

Iteration 1: Learning a new rule. Remaining positive: 1
  Considering literals for current rule (body: [])
  Added literal: ('Dependent', 'X'), Gain: 9.255
Learned rule: [('Dependent', 'X')]
  Covered 1 positive examples. 0 remaining.

FOLIO LEARNED RULES
Dependent(X) :- Dependent(X)


## Step 39 — Extract All Individuals from FOLIO Knowledge Base

To apply FOIL, especially for unary predicates, we need a complete set of all individuals (constants) present in our knowledge base. This allows us to define the universe for our positive and negative examples. We'll iterate through all facts in `folio_knowledge_base` and collect every argument that is not a variable (i.e., not a single lowercase letter like 'x', 'y', 'z').

```python
all_individuals = set()

for predicate, facts_set in folio_knowledge_base.items():
    for fact_args in facts_set:
        for arg in fact_args:
            if not is_variable_in_folio(arg):
                all_individuals.add(arg)

print(f"Total unique individuals found: {len(all_individuals)}")
print(list(all_individuals)[:10]) # Print first 10 for inspection
```

In [66]:
all_individuals = set()

for predicate, facts_set in folio_knowledge_base.items():
    for fact_args in facts_set:
        for arg in fact_args:
            if not is_variable_in_folio(arg):
                all_individuals.add(arg)

print(f"Total unique individuals found: {len(all_individuals)}")
print(list(all_individuals)[:10]) # Print first 10 for inspection

Total unique individuals found: 611
['neocrepidoderacorpulenta', 'year2016', 'monhoff', 'abc', 'jesse', 'cancerBiology', 'boapaymentcards', 'northamerica', 'matsOdell', 'nutter']


## Step 38 — Build a FOLIO Parser for `premises-FOL`

Now that we have a working FOIL implementation on a synthetic dataset, the next step is to adapt it to the full FOLIO dataset. This involves parsing the complex First-Order Logic (FOL) expressions found in the `premises-FOL` field.

Our goal here is to:
1. **Extract all atomic predicates** (e.g., `Drinks(x)`, `Student(rina)`, `¬Unaware(x)`) including their names, arguments (variables or constants), and whether they are negated.
2. **Populate a `knowledge_base`** with *ground facts* (atomic predicates with only constants as arguments) extracted from all `premises-FOL` expressions across the dataset.
3. **Identify all unique predicate signatures** (predicate name and arity) to later generate candidate literals for the FOIL algorithm.

We will define a function `extract_all_atomic_predicates_from_fol` that uses regular expressions to find these atomic units. We'll also define `is_variable_in_folio` to help distinguish between variables (`x`, `y`, `z`) and constants (`rina`, `phoenix`).

Then, `populate_knowledge_base_from_premises` will iterate through all records and their `premises-FOL` to build our global `knowledge_base` and collect all predicate signatures.

In [64]:
import re
from collections import defaultdict

def extract_all_atomic_predicates_from_fol(fol_string):
    """
    Extracts all atomic predicates (including arguments and negation) from a FOL expression string.
    Returns a list of tuples: (predicate_name, (arg1, arg2, ...), negated (bool))
    Handles patterns like `¬?PredicateName(arg1, arg2, ...)`.
    """
    extracted_predicates = []
    # Regex to find patterns like `¬?PredicateName(arg1, arg2, ...)`
    # Group 1: Negation (¬ or empty)
    # Group 2: PredicateName
    # Group 3: Arguments string (e.g., "x", "rina", "x, y")
    pattern = r'(¬?)([A-Z][A-Za-z0-9_]*)\(([^)]*)\)'
    matches = re.findall(pattern, fol_string)

    for neg_prefix, pred_name, args_str in matches:
        # Split arguments and clean them up
        args = tuple(arg.strip() for arg in args_str.split(',') if arg.strip())
        extracted_predicates.append((pred_name, args, True if neg_prefix == '¬' else False))
    return extracted_predicates

def is_variable_in_folio(arg):
    """
    Determines if an argument string represents a variable in FOLIO's context.
    Variables are typically single lowercase letters (x, y, z) or uppercase for FOIL's internal representation.
    """
    return arg.islower() and len(arg) == 1 # Common FOL variable convention

def populate_knowledge_base_from_premises(records):
    """
    Populates a knowledge base with ground facts extracted from all premises-FOL.
    Also collects all unique predicate signatures (name, arity).
    """
    kb = defaultdict(set)
    all_predicates_signatures = set() # To store (predicate_name, arity)

    for record in records:
        for premise_fol_string in record["premises-FOL"]:
            atomic_predicates = extract_all_atomic_predicates_from_fol(premise_fol_string)

            for pred_name, args, is_negated in atomic_predicates:
                all_predicates_signatures.add((pred_name, len(args)))

                # Add only ground facts (no variables, no negation) to the KB directly
                # A fact is ground if all its arguments are constants.
                if not is_negated and all(not is_variable_in_folio(arg) for arg in args):
                    kb[pred_name].add(args)
    return kb, all_predicates_signatures

# Example usage with the 'records' variable from previous steps
folio_knowledge_base, folio_predicate_signatures = populate_knowledge_base_from_premises(records)

print("--- Extracted FOLIO Knowledge Base (first 5 relations) ---")
for i, (pred, facts_set) in enumerate(folio_knowledge_base.items()):
    if i >= 5: break
    print(f"'{pred}': {list(facts_set)[:5]}...")

print("\n--- Extracted FOLIO Predicate Signatures (first 10) ---")
print(list(folio_predicate_signatures)[:10])

--- Extracted FOLIO Knowledge Base (first 5 relations) ---
'Student': [('joe',), ('john',), ('rina',)]...
'Unaware': [('rina',)]...
'Dependent': [('rina',)]...
'Czech': [('miroslavfiedler',), ('miroslav',)]...
'ChoralConductor': [('miroslav',)]...

--- Extracted FOLIO Predicate Signatures (first 10) ---
[('Play', 1), ('Matcha', 1), ('ProfessionalTennisPlayer', 1), ('GoodPerformance', 1), ('Magazine', 1), ('Entrepreneurs', 1), ('WatchTVInCinema', 1), ('PrefersCoolAt', 2), ('PlotsToSwallowUp', 2), ('Huron', 1)]
